This script processes supplements from [Dweck et al. 2018](https://www.cell.com/cell-reports/abstract/S2211-1247(18)30663-6): 
- Mapping between chemical to receptors are in firing rates (normalised); whereas fruit to receptors are in 'number of responses' (not normalised). 
- There are times when there are multiple receptors in the same columns. In that case those columns are duplicated, and unique names are given. 
- And there are times when multiple receptors map to the same glomerulus in the adult. In that case the max is taken: for one odour, if one receptor responds, then the ORNs in that glomerulus respond.

It saves: 
- adult_chem2glom.csv 
- larva_chem2or.csv 
- adult_fruit2glom.csv 
- larva_fruit2or.csv

In [1]:
import pandas as pd
import numpy as np

# pre-processed DoOR data

In [2]:
# preprocessed DoOR data 
door = pd.read_csv('data/DoOR/processed_door_adult.csv')
door

,glomerulus,sfr_SFR,water_XLYOFNOQVPJJNP-UHFFFAOYSA-N,ammonium hydroxide_VHUUQVKOLVNVRT-UHFFFAOYSA-N,putrescine_KIDHWZJUCRJVML-UHFFFAOYSA-N,cadaverine_VHRGRCVQAFMJIZ-UHFFFAOYSA-N,ammonia_QGZKDVFQNNGYKY-UHFFFAOYSA-N,ethanolamine_HZAXFHJVJLSVMW-UHFFFAOYSA-N,heptylamine_WJYIASZWHGOTOU-UHFFFAOYSA-N,isoamylamine_BMFVGAAISNGQNM-UHFFFAOYSA-N,...,(Z)-7-tricosene_IRYCVIRCWSSJOW-SQFISAMPSA-N,(Z)-9-tricosene_IGOWHGRNPLFNDJ-ZPHPHTNESA-N,(Z)-7-pentacosene_RORWYUWDGGVNRJ-SQFISAMPSA-N,"(Z,Z)-7,11-pentacosadiene_PJVWXEURPGIPNW-ADYYPQGGSA-N","(Z,Z)-7,11-heptacosadiene_RJYQFYALHBHYMG-ADYYPQGGSA-N","(Z,Z)-7,11-octacosadiene_KJSVCIRMHUQNGQ-ADYYPQGGSA-N","(Z,Z)-7,11-nonacosadiene_KWUWRILZYFCPRI-ADYYPQGGSA-N",methyl laurate_UQDUPQYQJKYHQI-UHFFFAOYSA-N,methyl palmitate_FLIACVVOZYBSBS-UHFFFAOYSA-N,palmitoleic acid_SECPZKHBENQXJG-FPLPWBNLSA-N
0,D,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,DA1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,DA2,0.064859,0.017059,0.000000,0.306816,0.025091,0.305932,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,DA3,0.019317,0.000000,0.015024,0.015024,0.004293,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,DA4l,0.095284,0.000000,0.090697,0.032186,0.048279,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,DA4m,0.048452,0.000000,0.056635,0.063791,0.051927,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,DC1,0.100069,0.000000,0.097216,0.080654,0.085971,0.000000,0.000000,0.037799,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,DC2,0.079803,0.197693,0.123406,0.153453,0.000000,0.118779,0.128289,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,DC3,0.078341,0.000000,0.000000,0.000000,0.000000,0.170507,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,DC4,0.000000,0.854963,0.028357,0.326230,0.293186,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
receptor2glom = pd.read_csv(
    'data/DoOR/door_receptor_mappings.csv', index_col=0)
receptor2glom

,receptor,sensillum,OSN,glomerulus,co.receptor,coexpressing,related1,related2,related3,related4,related5,related6,Ors,sensillum.type,adult,larva,dataset.existing,comment,code,code.OSN
1,?,?,?,VA7m,?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,?,NaN,NaN,NaN,False,NaN,VA7m,NaN
2,Or42b,ab1,ab1A,DM1,Orco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Or42b,antennal basiconic,True,True,True,NaN,DM1,ab1A
3,Or92a,ab1,ab1B,VA2,Orco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Or92a,antennal basiconic,True,NaN,True,NaN,VA2,ab1B
4,Gr21a.Gr63a,ab1,ab1C,V,NaN,Gr21a+Gr63a,NaN,NaN,NaN,NaN,NaN,NaN,Gr21a+Gr63a,antennal basiconic,True,True,True,NaN,V,ab1C
5,Or10a,ab1,ab1D,DL1,Orco,Gr10a,Gr10a,NaN,NaN,NaN,NaN,NaN,Or10a+Gr10a,antennal basiconic,True,NaN,True,NaN,DL1,ab1D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,Or83a,NaN,NaN,NaN,Orco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,NaN,NaN,NaN
93,Or83b,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,False,the Orco co-receptor expressed together with a...,NaN,NaN
94,Or85c,NaN,NaN,NaN,Orco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,NaN,NaN,NaN
95,Or94a,NaN,NaN,NaN,Orco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,NaN,NaN,NaN


In [4]:
door_chemical_meta = pd.read_csv(
    'data/DoOR/door_chemical_meta.csv', index_col=0)
door_chemical_meta

,Class,Name,InChIKey,CID,CAS,Stensmyr.2003.WT,Schmuker.2007.TR,Dobritsa.2003.WT,Bruyne.2001.WT,Bruyne.2010.WT,Marshall.2010.WT,Hallem.2004.WT
1,NaN,sfr,SFR,SFR,SFR,NaN,1.0,NaN,1.0,0.0,0.000,NaN
2,other,water,XLYOFNOQVPJJNP-UHFFFAOYSA-N,962,7732-18-5,NaN,NaN,NaN,NaN,NaN,5.714,NaN
3,amine,ammonium hydroxide,VHUUQVKOLVNVRT-UHFFFAOYSA-N,14923,1336-21-6,NaN,NaN,NaN,NaN,NaN,2.714,NaN
4,amine,putrescine,KIDHWZJUCRJVML-UHFFFAOYSA-N,1045,110-60-1,NaN,0.0,NaN,NaN,NaN,5.667,NaN
5,amine,cadaverine,VHRGRCVQAFMJIZ-UHFFFAOYSA-N,273,462-94-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
689,NaN,"(Z,Z)-7,11-octacosadiene",KJSVCIRMHUQNGQ-ADYYPQGGSA-N,14464905,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
690,NaN,"(Z,Z)-7,11-nonacosadiene",KWUWRILZYFCPRI-ADYYPQGGSA-N,14367348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
691,NaN,methyl laurate,UQDUPQYQJKYHQI-UHFFFAOYSA-N,8139,111-82-0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
692,NaN,methyl palmitate,FLIACVVOZYBSBS-UHFFFAOYSA-N,8181,112-39-0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Dweck 2018 data

### supp2  chemical-OR
Table S1. List of identified active compounds in both larval and adult Drosophila (related to Fig. 2)  
Numbers represent  maximum frequency (Hz) responses (mean ± SD) to  10-4 concentration of synthetic standards of the active copmpounds (n = 2-3recordings). Synthetic standards were applied via GC in GC-SSR experiments, which accounts for the very accurate stimulus delivery and the corresponding low standard deviations.


In [5]:
chem2or = pd.read_excel('data/Dweck2018/1-s2.0-S2211124718306636-mmc2.xlsx', 
                      skiprows=7)
chem2or.rename(columns={'Unnamed: 0': 'chemical'}, inplace=True)

# there are columns where the entire column is filled with NaNs. 
# I'll take this to mean that authors tried this receptor, but it didn't respond to any chemicals 
# therefore we will keep them, and replace all NaNs with 0. 
# and remove empty strings 
chem2or.replace({' ': np.nan}, inplace=True)

chem2or

,chemical,CAS number,Or2a,Or7a,Or9a,Or10a,Or13a,Or19a,Or22a/Or22b,Or22c,...,Or83c,Or85a,Or85b,Or85c,Or85d,Or88a,Or92a,Or94a,Or94b,Or98a
0,(-)-(E)-Caryophyllene,87-44-5,NaN,NaN,NaN,NaN,NaN,71 ±4,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,(-)-a-Copaene,3856-25-5,NaN,NaN,NaN,NaN,NaN,62 ± 2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(-)-Camphor,464-48-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(-)-Menthone,14073-97-3,NaN,NaN,NaN,NaN,NaN,60 ± 0.4,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,(-)-Terpinen-4-ol,20126-76-5,NaN,NaN,NaN,NaN,NaN,42 ± 1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,Veratrole,91-16-7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,187 ± 12,NaN,NaN
112,β-Citronellol,106-22-9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,β-Elemene,33880-83-0,NaN,NaN,NaN,NaN,NaN,43 ± 3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,β-Ionone,79-77-6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# tidy up the numbers: take the mean only 
for col in chem2or.columns:
    if col not in ['chemical','CAS number']:
        vs = []
        for v in chem2or[col]: 
            if pd.isna(v): 
                vs.append(np.nan)
            elif type(v) == int: 
                vs.append(int(v))
            elif (type(v) == np.float64) or (type(v) == float): 
                vs.append(int(v))
            else: 
                vs.append(int(v.split(' ±')[0]))
        chem2or[col] = vs
chem2or

,chemical,CAS number,Or2a,Or7a,Or9a,Or10a,Or13a,Or19a,Or22a/Or22b,Or22c,...,Or83c,Or85a,Or85b,Or85c,Or85d,Or88a,Or92a,Or94a,Or94b,Or98a
0,(-)-(E)-Caryophyllene,87-44-5,NaN,NaN,NaN,NaN,NaN,71.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,(-)-a-Copaene,3856-25-5,NaN,NaN,NaN,NaN,NaN,62.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(-)-Camphor,464-48-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(-)-Menthone,14073-97-3,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,(-)-Terpinen-4-ol,20126-76-5,NaN,NaN,NaN,NaN,NaN,42.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,Veratrole,91-16-7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,187.0,NaN,NaN
112,β-Citronellol,106-22-9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,β-Elemene,33880-83-0,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,β-Ionone,79-77-6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
chem2or.columns.unique()

Index(['chemical', 'CAS number', 'Or2a', 'Or7a', 'Or9a', 'Or10a', 'Or13a',
       'Or19a', 'Or22a/Or22b', 'Or22c', 'Or23a', 'Or24a', 'Or30a', 'Or33b',
       'Or33c/Or85e', 'Or35a', 'Or42a', 'Or42b', 'Or43a', 'Or43b', 'Or45a',
       'Or45b', 'Or46a', 'Or47a', 'Or47b', 'Or49a/Or85f', 'Or49b', 'Or56a',
       'Or59a', 'Or59b', 'Or59c', 'Or65a', 'Or67a', 'Or67b', 'Or67c', 'Or67d',
       'Or69a', 'Or71a', 'Or74a', 'Or82a', 'Or83c', 'Or85a', 'Or85b', 'Or85c',
       'Or85d', 'Or88a', 'Or92a', 'Or94a', 'Or94b', 'Or98a'],
      dtype='object')

In [8]:
# multiple receptors are mixed into the same column, which means they can't be directly mapped to glomerulus. 
# let's duplicate those columns 
chem2or.loc[:,['Or22b']] = chem2or['Or22a/Or22b'].copy()
chem2or.loc[:, ['Or85e']] = chem2or['Or33c/Or85e'].copy()
chem2or.rename(columns={
    'Or22a/Or22b': 'Or22a', 
    'Or33c/Or85e': 'Or33c'
}, inplace=True)
chem2or.fillna(0, inplace = True)

chem2or

,chemical,CAS number,Or2a,Or7a,Or9a,Or10a,Or13a,Or19a,Or22a,Or22c,...,Or85b,Or85c,Or85d,Or88a,Or92a,Or94a,Or94b,Or98a,Or22b,Or85e
0,(-)-(E)-Caryophyllene,87-44-5,0.0,0.0,0.0,0.0,0.0,71.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,(-)-a-Copaene,3856-25-5,0.0,0.0,0.0,0.0,0.0,62.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,(-)-Camphor,464-48-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39.0
3,(-)-Menthone,14073-97-3,0.0,0.0,0.0,0.0,0.0,60.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,(-)-Terpinen-4-ol,20126-76-5,0.0,0.0,0.0,0.0,0.0,42.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,Veratrole,91-16-7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,187.0,0.0,0.0,0.0,0.0
112,β-Citronellol,106-22-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
113,β-Elemene,33880-83-0,0.0,0.0,0.0,0.0,0.0,43.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
114,β-Ionone,79-77-6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0


In [9]:
# to normalise, dividing by the max of all receptors 
# assuming that receptors are more or less similar, 
# and that odours are not super complete to reach the whole range for all receptors 
chem2or_values = chem2or.set_index(['chemical','CAS number'])
chem2or_values = chem2or_values / chem2or_values.max().max()
chem2or = chem2or_values.reset_index()
chem2or

,chemical,CAS number,Or2a,Or7a,Or9a,Or10a,Or13a,Or19a,Or22a,Or22c,...,Or85b,Or85c,Or85d,Or88a,Or92a,Or94a,Or94b,Or98a,Or22b,Or85e
0,(-)-(E)-Caryophyllene,87-44-5,0.0,0.0,0.0,0.0,0.0,0.331776,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
1,(-)-a-Copaene,3856-25-5,0.0,0.0,0.0,0.0,0.0,0.289720,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
2,(-)-Camphor,464-48-2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.182243
3,(-)-Menthone,14073-97-3,0.0,0.0,0.0,0.0,0.0,0.280374,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
4,(-)-Terpinen-4-ol,20126-76-5,0.0,0.0,0.0,0.0,0.0,0.196262,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,Veratrole,91-16-7,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.873832,0.0,0.0,0.0,0.000000
112,β-Citronellol,106-22-9,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
113,β-Elemene,33880-83-0,0.0,0.0,0.0,0.0,0.0,0.200935,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
114,β-Ionone,79-77-6,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.238318


In [10]:
# map chemicals to adult glomeruli 
receptor_to_glom = dict(zip(receptor2glom.receptor[~receptor2glom.glomerulus.isna()], 
receptor2glom.glomerulus[~receptor2glom.glomerulus.isna()]))

adult_chem2or = pd.concat([chem2or[['chemical','CAS number']], 
# only get the receptors that can be mapped to the glomerulus according to Munch and Galizia 
chem2or.loc[:,chem2or.columns.isin(receptor_to_glom)]], axis = 1)
adult_chem2or.columns = [receptor_to_glom[r] if r in receptor_to_glom else r for r in adult_chem2or.columns] 

adult_chem2or.columns

Index(['chemical', 'CAS number', 'DA4m', 'DL5', 'VM3', 'DL1', 'DC2', 'DC1',
       'DM2', 'DA3', 'DM5+DM3', 'VC1', 'VC3', 'VM7d', 'DM1', 'DA4l', 'VM2',
       'VA7l', 'DM3', 'VA1v', 'VA5', 'DA2', 'DM4', 'VM7v', 'DL3', 'DM6', 'VA3',
       'VC4', 'DA1', 'D', 'VC2', 'VA6', 'DC3', 'DM5', 'VM5d', 'VA4', 'VA1d',
       'VA2', 'VM5v', 'DM2', 'VC1'],
      dtype='object')

In [11]:
# DM3 and DM5 appearing multiple times 
# glomeruli map back to the receptors. So e.g. DM3 has multiple receptors. 
# DM3 ORNs activate when either receptor activates. Therefore we can take the max 
adult_chem2or.loc[:,'DM3'] = adult_chem2or[['DM3','DM5+DM3']].max(axis = 1)
adult_chem2or.loc[:,'DM5'] = adult_chem2or[['DM5','DM5+DM3']].max(axis = 1)
adult_chem2or.drop('DM5+DM3', axis = 1, inplace = True)

adult_chem2or

,chemical,CAS number,DA4m,DL5,VM3,DL1,DC2,DC1,DM2,DA3,...,VA6,DC3,DM5,VM5d,VA4,VA1d,VA2,VM5v,DM2,VC1
0,(-)-(E)-Caryophyllene,87-44-5,0.0,0.0,0.0,0.0,0.0,0.331776,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,(-)-a-Copaene,3856-25-5,0.0,0.0,0.0,0.0,0.0,0.289720,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2,(-)-Camphor,464-48-2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.182243
3,(-)-Menthone,14073-97-3,0.0,0.0,0.0,0.0,0.0,0.280374,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,(-)-Terpinen-4-ol,20126-76-5,0.0,0.0,0.0,0.0,0.0,0.196262,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,Veratrole,91-16-7,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
112,β-Citronellol,106-22-9,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
113,β-Elemene,33880-83-0,0.0,0.0,0.0,0.0,0.0,0.200935,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
114,β-Ionone,79-77-6,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.238318


In [12]:
#how many are shared with DoOR? 
len(set(adult_chem2or['CAS number']) & set(door_chemical_meta.CAS))

88

In [13]:
#which glomerulus is missing in Dweck et al. compared to DoOR? 
set(door.glomerulus) - set(adult_chem2or.columns)

{'DC4',
 'DL2d',
 'DL2v',
 'DL4',
 'DP1l',
 'DP1m',
 'V',
 'VC5',
 'VL1',
 'VL2a',
 'VL2p',
 'VM1',
 'VM4'}

In [14]:
# any present in Dweck but not DoOR? 
set(adult_chem2or.columns) - set(door.glomerulus)

{'CAS number', 'chemical'}

In [15]:
# further tidy up to have glomeruli in the rows 
adult_chem2or = adult_chem2or.drop('CAS number', axis = 1).set_index('chemical').T
adult_chem2or.index.name = 'glomerulus'
adult_chem2or

chemical,(-)-(E)-Caryophyllene,(-)-a-Copaene,(-)-Camphor,(-)-Menthone,(-)-Terpinen-4-ol,(+)-Limonene oxide,(±)-6-Methyl-5-hepten-2-ol,(±)-Ethyl 3-acetoxy butyrate,(±)-Sabinene,(E)-2-Hexen-1-yl propionate,...,Phenethyl isobutyrate,Phenethyl propionate,Prenyl acetate,Terpinolene,Valencene,Veratrole,β-Citronellol,β-Elemene,β-Ionone,β-Myrcene
glomerulus,,,,,,,,,,,,,,,,,,,,,
DA4m,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
DL5,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
VM3,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
DL1,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
DC2,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
DC1,0.331776,0.28972,0.000000,0.280374,0.196262,0.275701,0.000000,0.000000,0.163551,0.000000,...,0.000000,0.000000,0.098131,0.172897,0.21028,0.000000,0.00000,0.200935,0.000000,0.093458
DM2,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
DA3,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
VC1,0.000000,0.00000,0.182243,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.238318,0.000000


In [16]:
# larva
larva_chem2or = pd.concat([chem2or[['chemical','CAS number']], 
chem2or.loc[:, chem2or.columns.isin(receptor2glom.receptor[receptor2glom.larva.isin([True])])]], axis = 1)
larva_chem2or = larva_chem2or.drop('CAS number', axis = 1).set_index('chemical').T
larva_chem2or.index.name = 'receptor'
larva_chem2or

chemical,(-)-(E)-Caryophyllene,(-)-a-Copaene,(-)-Camphor,(-)-Menthone,(-)-Terpinen-4-ol,(+)-Limonene oxide,(±)-6-Methyl-5-hepten-2-ol,(±)-Ethyl 3-acetoxy butyrate,(±)-Sabinene,(E)-2-Hexen-1-yl propionate,...,Phenethyl isobutyrate,Phenethyl propionate,Prenyl acetate,Terpinolene,Valencene,Veratrole,β-Citronellol,β-Elemene,β-Ionone,β-Myrcene
receptor,,,,,,,,,,,,,,,,,,,,,
Or2a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
Or7a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
Or9a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
Or22c,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
Or24a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
Or30a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
Or33b,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
Or35a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.373832,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
Or42a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [17]:
adult_chem2or.to_csv('data/Dweck2018/adult_chem2glom.csv')
larva_chem2or.to_csv('data/Dweck2018/larva_chem2or.csv')

### supp3 chemical-fruit
Table S2. List of identified active compounds in 34 fruit headspaces (related to Fig. 4)  
Numbers represent normalized relative abundance of the active compounds in each tested fruit extract. In those columns, where no compound reaches the value 1, the compound with highest abundance was not physiologically active


In [18]:
chem2fruit = pd.read_excel('data/Dweck2018/1-s2.0-S2211124718306636-mmc3.xlsx', skiprows=3)
chem2fruit

,Unnamed: 0,CAS number,Apple,Apricot,Avocado,Banana,Blackberry,Blueberry,Cactus fig,Currant,...,Pear,Physalisus,Pineapple,Plums,Pomegranate,Raspberry,Strawberry,Java plum,Af. breadfruit,Watermelon
0,(-)-(E)-Caryophyllene,87-44-5,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.053104,NaN,0.003376,0.482631,0.011736,NaN,0.017498,NaN,0.001189
1,(-)-a-Copaene,3856-25-5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(-)-Camphor,464-48-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.02028,...,0.000259,NaN,NaN,NaN,0.003417,0.003485,0.000310,0.002204,0.019367,0.002581
3,(-)-Menthone,14073-97-3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.401326,NaN
4,(-)-Terpinen-4-ol,20126-76-5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.111157,NaN,NaN,NaN,NaN,NaN,0.173440,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,Veratrole,91-16-7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.003143,NaN,NaN,NaN,NaN,NaN,0.002965,NaN
112,β-Citronellol,106-22-9,0.000745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,β-Elemene,33880-83-0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.019637,NaN,NaN,NaN,0.001705,NaN,NaN,NaN,0.037121
114,β-Ionone,79-77-6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.628531,NaN,NaN,NaN,NaN


### supp4: fruit-OR
Table S3. List of fruit-specific activity of all measured responses of neurons expressing larval or adult olfactory receptors (related to Fig. 4)
Total number of responses of individual larval and adult receptors when tested with the individual  fruit headspaces.

In [19]:
supp4 = pd.read_excel(
    'data/Dweck2018/1-s2.0-S2211124718306636-mmc4.xlsx', skiprows=3)
supp4

,Unnamed: 0,Unnamed: 1,Or2a,Or7a,Or13a,Or22c,Or24a,Or30a,Or33b,Or35a,...,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49
0,NaN,African mango,0,1,2,4,1,1,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,African breadfruit,0,4,5,3,1,1,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,African nutmeg,0,1,1,2,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Apple,0,5,2,3,1,1,0,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Apricot,0,0,1,1,1,1,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,NaN,Plums,0,0,2,0,1,2,1,0,...,0,0,0,0,0,0,0,0,0,0
71,NaN,Pomegranate,0,0,1,0,2,1,1,0,...,0,0,0,0,0,0,0,0,0,0
72,NaN,Raspberry,0,2,2,0,2,4,4,0,...,0,0,0,0,0,0,0,0,0,0
73,NaN,Strawberry,0,1,2,0,1,1,8,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
larva_fruit2or = supp4.iloc[:34, 1:23]
# based on receptor2glom, I'll take column 'Or42b/ab1' to mean 'either Or42b or sensillum ab1' 
# and approximate it with Or42b
larva_fruit2or.rename(columns={'Unnamed: 1': 'fruit', 'Or42b/ab1': 'Or42b'}, inplace=True)
larva_fruit2or.columns

Index(['fruit', 'Or2a', 'Or7a', 'Or13a', 'Or22c', 'Or24a', 'Or30a', 'Or33b',
       'Or35a', 'Or42a', 'Or42b', 'Or45a', 'Or45b', 'Or47a', 'Or49a', 'Or59a',
       'Or67b', 'Or74a', 'Or82a', 'Or85c', 'Or94a', 'Or94b'],
      dtype='object')

In [21]:
larva_fruit2or = larva_fruit2or.set_index("fruit").T
larva_fruit2or.index.name = "receptor"
larva_fruit2or

fruit,African mango,African breadfruit,African nutmeg,Apple,Apricot,Avocado,Banana,Blackberry,Blueberry,Cactus fig,...,Passionfruit,Peach,Pear,Physalis,Pineapple,Plums,Pomegranate,Raspberry,Strawberry,Watermelon
receptor,,,,,,,,,,,,,,,,,,,,,
Or2a,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Or7a,1,4,1,5,0,0,1,0,1,1,...,0,2,5,1,0,0,0,2,1,2
Or13a,2,5,1,2,1,1,2,3,1,2,...,3,2,3,2,2,1,2,2,1,1
Or22c,4,3,2,3,1,0,1,2,0,2,...,5,0,2,3,3,1,0,1,4,2
Or24a,1,1,1,1,1,1,1,0,0,1,...,1,1,1,2,1,1,1,1,1,1
Or30a,1,1,1,1,1,1,1,1,0,0,...,1,1,1,0,1,1,1,1,1,1
Or33b,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Or35a,1,2,0,8,2,1,4,1,1,3,...,0,2,3,3,0,3,0,2,5,2
Or42a,1,4,4,4,2,0,2,0,0,1,...,5,1,3,3,3,0,1,1,3,1


In [22]:
adult_fruit2or = supp4.iloc[40:, 1:].reset_index(drop='index')
adult_fruit2or.columns = adult_fruit2or.loc[0]
adult_fruit2or.rename(columns={np.nan: 'fruit'}, inplace=True)
adult_fruit2or = adult_fruit2or.iloc[1:,:]

# some columns are filled with 0s 
# I assume this means that the authors tested these receptors, but they didn't activate in response to fruits 
# so they still contain information, and are to be kept. 

adult_fruit2or.columns

Index(['fruit', 'Or2a', 'Or7a', 'Or9a', 'Or10a', 'Or13a', 'Or19a',
       'Or22a/Or22b', 'Or23a', 'Or33c/Or85e', 'Or35a', 'Or42a', 'Or42b',
       'Or43a', 'Or43b', 'Or46a', 'Or47a', 'Or47b', 'Or49a/Or85f', 'Or49b',
       'Or56a', 'Or59b', 'Or59c', 'Or65a', 'Or67a', 'Or67b', 'Or67c', 'Or67d',
       'Or69a', 'Or71a', 'Or82a', 'Or83c', 'Or85a', 'Or85b', 'Or85d', 'Or88a',
       'Or92a', 'Or98a', 'Gr63a/Gr21a', 'Ir31a', 'Ir41a/Ir76b', 'Ir75a',
       'Ir75c/Ir75b/Ir75a', 'Ir75d', 'Ir75d', 'Ir75d', 'Ir76a/Ir76b', 'Ir84a',
       'Ir92a/Ir76b'],
      dtype='object', name=0)

In [23]:
colsums = adult_fruit2or.sum()
colsums[colsums == 0].index

Index(['Or2a', 'Or23a', 'Or43a', 'Or47b', 'Or49a/Or85f', 'Or56a', 'Or65a',
       'Or67d', 'Or88a', 'Gr63a/Gr21a', 'Ir31a', 'Ir41a/Ir76b', 'Ir75a',
       'Ir75c/Ir75b/Ir75a', 'Ir75d', 'Ir75d', 'Ir75d', 'Ir76a/Ir76b', 'Ir84a',
       'Ir92a/Ir76b'],
      dtype='object', name=0)

In [24]:
adult_fruit2or = adult_fruit2or.loc[:, ~
                                    adult_fruit2or.columns.duplicated()].copy()
# duplicate columns with multiple receptors 
adult_fruit2or.loc[:,['Or22b']] = adult_fruit2or['Or22a/Or22b']
adult_fruit2or.loc[:, ['Or85e']] = adult_fruit2or['Or33c/Or85e'].copy()
adult_fruit2or.rename(columns = {
    'Or22a/Or22b': 'Or22a', 
    'Or33c/Or85e': 'Or33c'
}, inplace = True)
# after manual checking, all other columns with '/' all have 0s in all the constituents 
adult_fruit2or.loc[:, ['Or49a', 'Or85f', 'Gr63a',
                       'Gr21a', 'Ir41a', 'Ir76b', 'Ir75c', 'Ir75b', 'Ir76a', 'Ir92a']] = 0
adult_fruit2or.drop(['Or49a/Or85f','Gr63a/Gr21a', 'Ir41a/Ir76b', 'Ir75c/Ir75b/Ir75a', 'Ir76a/Ir76b', 'Ir92a/Ir76b'], axis = 1, inplace = True)

# and transpose
adult_fruit2or = adult_fruit2or.set_index('fruit').T
adult_fruit2or.index.name = 'receptor'
adult_fruit2or

fruit,African breadfruit,African mango,African nutmeg,Apple,Apricot,Avocado,Banana,Blackberry,Blueberry,Cactus fig,...,Passionfruit,Peach,Pear,Physalis,Pineapple,Plums,Pomegranate,Raspberry,Strawberry,Watermelon
receptor,,,,,,,,,,,,,,,,,,,,,
Or2a,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Or7a,4,1,1,5,0,0,1,0,1,1,...,0,2,5,1,0,0,0,2,1,2
Or9a,2,2,1,1,1,0,3,1,2,3,...,4,1,1,3,3,2,1,2,2,1
Or10a,0,4,3,0,1,0,0,0,0,1,...,3,0,0,2,1,0,0,0,0,0
Or13a,5,2,1,2,1,1,2,3,1,2,...,3,2,3,2,2,1,2,2,1,1
Or19a,4,5,12,1,0,3,2,0,0,0,...,6,0,1,6,0,2,1,4,1,3
Or22a,11,8,4,9,1,1,3,7,4,8,...,5,4,4,5,8,1,1,4,8,1
Or23a,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Or33c,3,2,6,2,2,0,0,0,0,0,...,5,3,0,5,2,0,0,5,2,2


In [25]:
# try to map to glomeruli
adult_fruit2or.index.map(receptor_to_glom)

Index([  'DA4m',    'DL5',    'VM3',    'DL1',    'DC2',    'DC1',    'DM2',
          'DA3',    'VC1',    'VC3',   'VM7d',    'DM1',   'DA4l',    'VM2',
         'VA7l',    'DM3',   'VA1v',    'VA5',    'DA2',    'DM4',   'VM7v',
          'DL3',    'DM6',    'VA3',    'VC4',    'DA1',      'D',    'VC2',
          'VA6',    'DC3',    'DM5',   'VM5d',    'VA4',   'VA1d',    'VA2',
         'VM5v',   'VL2p',   'DP1l',    'VL1',   'VL2a',    'DM2',    'VC1',
          'DL4',    'DL4',      nan,      nan,    'VC5',      nan, 'DL2d/v',
       'DL2d/v',    'VM4',    'VM1'],
      dtype='object', name='receptor')

In [26]:
# which receptors are not mapped to a glomerulus?
adult_fruit2or.index[~adult_fruit2or.index.isin(receptor_to_glom)]

Index(['Gr63a', 'Gr21a', 'Ir76b'], dtype='object', name='receptor')

In [27]:
adult_fruit2or.loc['Ir76b',:].sum()

0

Gr63a and Gr21a are shwon to innervate glomerulus V in [Kwon et al. 2007](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1805529/). Ir76b innervates multiple glomeruli [Task et al. 2022](https://elifesciences.org/articles/72599) (table 3), but the values are all 0 here. So let's ignore. 

In [28]:
receptor_to_glom.update({'Gr63a':'V', 'Gr21a':'V'})
adult_fruit2or.index = adult_fruit2or.index.map(receptor_to_glom)
adult_fruit2or.index.name = 'glomerulus'
adult_fruit2or

fruit,African breadfruit,African mango,African nutmeg,Apple,Apricot,Avocado,Banana,Blackberry,Blueberry,Cactus fig,...,Passionfruit,Peach,Pear,Physalis,Pineapple,Plums,Pomegranate,Raspberry,Strawberry,Watermelon
glomerulus,,,,,,,,,,,,,,,,,,,,,
DA4m,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DL5,4,1,1,5,0,0,1,0,1,1,...,0,2,5,1,0,0,0,2,1,2
VM3,2,2,1,1,1,0,3,1,2,3,...,4,1,1,3,3,2,1,2,2,1
DL1,0,4,3,0,1,0,0,0,0,1,...,3,0,0,2,1,0,0,0,0,0
DC2,5,2,1,2,1,1,2,3,1,2,...,3,2,3,2,2,1,2,2,1,1
DC1,4,5,12,1,0,3,2,0,0,0,...,6,0,1,6,0,2,1,4,1,3
DM2,11,8,4,9,1,1,3,7,4,8,...,5,4,4,5,8,1,1,4,8,1
DA3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
VC1,3,2,6,2,2,0,0,0,0,0,...,5,3,0,5,2,0,0,5,2,2


In [29]:
# multiple receptors are mapped to the same glomerulus. In this case we take the max:
# if any receptor activates, the glomerulus is activated.
adult_fruit2or = adult_fruit2or.groupby('glomerulus').max()
# now split 'DL2d/v' into 'DL2d' and 'DL2v'
adult_fruit2or.loc['DL2d',:] = adult_fruit2or.loc['DL2d/v', :].copy()
adult_fruit2or.rename(index={'DL2d/v': 'DL2v'}, inplace=True)

adult_fruit2or 
# note how now the 'NaN' row is gone (compared to the last code block) - probably because of the grouping by function 

fruit,African breadfruit,African mango,African nutmeg,Apple,Apricot,Avocado,Banana,Blackberry,Blueberry,Cactus fig,...,Passionfruit,Peach,Pear,Physalis,Pineapple,Plums,Pomegranate,Raspberry,Strawberry,Watermelon
glomerulus,,,,,,,,,,,,,,,,,,,,,
D,0,0,2,3,0,0,1,0,0,0,...,9,4,2,1,4,3,0,0,4,1
DA1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DA2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DA3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DA4l,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DA4m,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DC1,4,5,12,1,0,3,2,0,0,0,...,6,0,1,6,0,2,1,4,1,3
DC2,5,2,1,2,1,1,2,3,1,2,...,3,2,3,2,2,1,2,2,1,1
DC3,0,0,0,2,0,0,0,0,0,0,...,1,0,1,0,0,0,0,0,0,0


In [30]:
adult_fruit2or.to_csv('data/Dweck2018/adult_fruit2glom.csv')
larva_fruit2or.to_csv('data/Dweck2018/larva_fruit2or.csv')